# App Review Source Technical Validation - Run 1

This notebook is a clean Run 1 validation for public app review collection sources.

The goal is to test whether Google Play and iOS App Store public review sources can support a recurring ingestion workflow.

The validation focuses on:

- access method
- review volume
- available metadata fields
- pagination and batching
- duplicate handling
- freshness
- field consistency
- request failures
- exploratory data analysis
- data quality issues

Google Play is tested as the primary source. iOS App Store public RSS feed is tested as a secondary source.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os
from datetime import datetime

PROJECT_DIR = "/content/drive/MyDrive/app_review_source_validation"

folders = [
    "data/raw",
    "data/processed",
    "outputs/summaries",
    "outputs/figures",
    "reports"
]

for folder in folders:
    os.makedirs(os.path.join(PROJECT_DIR, folder), exist_ok=True)

RUN_DATE = datetime.now().strftime("%Y_%m_%d")
RUN_LABEL = "run1"

print("Project folder:", PROJECT_DIR)
print("Run date:", RUN_DATE)
print("Run label:", RUN_LABEL)

Project folder: /content/drive/MyDrive/app_review_source_validation
Run date: 2026_06_22
Run label: run1


In [ ]:
!pip install google-play-scraper pandas requests matplotlib langdetect

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.2/50.2 kB 2.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 19.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for langdetect: filename=langdetect-1.0.9-py3-none-any.whl size=993223 sha256=cdb2e7ffc80eb18c70535ffa73fead86974d205f485e2ced80b388d83d9e79ba
  Stored in directory: /root/.cache/pip/wheels/c1/67/88/e844b5b022812e15a52e4eaa38a1e709e99f06f6639d7e3ba7
Successfully built langdetect


In [ ]:
import pandas as pd
import requests
import time
import matplotlib.pyplot as plt

from datetime import datetime
from google_play_scraper import reviews, Sort
from langdetect import detect, DetectorFactory

DetectorFactory.seed = 0

In [ ]:
apps = [
    {
        "name": "YouTube",
        "google_package": "com.google.android.youtube",
        "ios_id": "544007664"
    },
    {
        "name": "TikTok",
        "google_package": "com.zhiliaoapp.musically",
        "ios_id": "835599320"
    },
    {
        "name": "Spotify",
        "google_package": "com.spotify.music",
        "ios_id": "324684580"
    }
]

countries = ["us", "gb", "ca", "au", "in"]

apps

[{'name': 'YouTube',
  'google_package': 'com.google.android.youtube',
  'ios_id': '544007664'},
 {'name': 'TikTok',
  'google_package': 'com.zhiliaoapp.musically',
  'ios_id': '835599320'},
 {'name': 'Spotify',
  'google_package': 'com.spotify.music',
  'ios_id': '324684580'}]

## 1. Google Play Review Collection

Google Play is tested as the primary source using the `google-play-scraper` Python package.

This path does not require Google Play Console owner access. For Run 1, I collected the newest US / English reviews for each app.

The test checks:

- whether public reviews can be collected
- whether batching works
- whether review volume is sufficient
- whether review IDs are available for duplicate handling
- whether the newest reviews are fresh
- whether metadata fields are usable for downstream analysis

In [ ]:
def collect_google_play(app_list, target_count=2000, country="us", lang="en"):
    rows = []
    summary = []

    for app in app_list:
        app_name = app["name"]
        package = app["google_package"]

        token = None
        total = 0
        failed = False
        error_message = None
        start = time.time()

        while total < target_count:
            try:
                batch, token = reviews(
                    package,
                    lang=lang,
                    country=country,
                    sort=Sort.NEWEST,
                    count=min(200, target_count - total),
                    continuation_token=token
                )

                if len(batch) == 0:
                    break

                for r in batch:
                    rows.append({
                        "source": "google_play",
                        "app_name": app_name,
                        "app_package": package,
                        "country": country,
                        "language": lang,
                        "review_id": r.get("reviewId"),
                        "user_name": r.get("userName"),
                        "review_text": r.get("content"),
                        "rating": r.get("score"),
                        "thumbs_up": r.get("thumbsUpCount"),
                        "review_created_version": r.get("reviewCreatedVersion"),
                        "review_date": r.get("at"),
                        "developer_reply": r.get("replyContent"),
                        "reply_date": r.get("repliedAt"),
                        "app_version": r.get("appVersion")
                    })

                total += len(batch)

                if token is None:
                    break

                time.sleep(0.3)

            except Exception as e:
                failed = True
                error_message = str(e)
                break

        runtime_seconds = round(time.time() - start, 2)

        summary.append({
            "source": "google_play",
            "app_name": app_name,
            "app_package": package,
            "country": country,
            "language": lang,
            "target_count": target_count,
            "collected_count": total,
            "failed": failed,
            "error_message": error_message,
            "runtime_seconds": runtime_seconds
        })

    return pd.DataFrame(rows), pd.DataFrame(summary)

In [ ]:
google_reviews, google_collection_summary = collect_google_play(apps, target_count=2000)

google_raw_path = os.path.join(PROJECT_DIR, f"data/raw/google_play_reviews_{RUN_LABEL}_{RUN_DATE}.csv")
google_summary_path = os.path.join(PROJECT_DIR, f"outputs/summaries/google_play_collection_summary_{RUN_LABEL}_{RUN_DATE}.csv")

google_reviews.to_csv(google_raw_path, index=False)
google_collection_summary.to_csv(google_summary_path, index=False)

display(google_collection_summary)

print("Saved raw data to:", google_raw_path)
print("Saved summary to:", google_summary_path)
print("Total Google Play reviews collected:", len(google_reviews))

,source,app_name,app_package,country,language,target_count,collected_count,failed,error_message,runtime_seconds
0,google_play,YouTube,com.google.android.youtube,us,en,2000,2000,False,None,6.15
1,google_play,TikTok,com.zhiliaoapp.musically,us,en,2000,2000,False,None,4.98
2,google_play,Spotify,com.spotify.music,us,en,2000,2000,False,None,4.96


Saved raw data to: /content/drive/MyDrive/app_review_source_validation/data/raw/google_play_reviews_run1_2026_06_22.csv
Saved summary to: /content/drive/MyDrive/app_review_source_validation/outputs/summaries/google_play_collection_summary_run1_2026_06_22.csv
Total Google Play reviews collected: 6000


## 2. Google Play Validation Summary

This section summarizes the Google Play Run 1 result.

The goal is to check review volume, review ID uniqueness, freshness, duplicate count, and metadata coverage for each app.

In [ ]:
google_validation_summary = (
    google_reviews
    .groupby("app_name")
    .agg(
        review_count=("review_id", "count"),
        unique_review_ids=("review_id", "nunique"),
        oldest_review=("review_date", "min"),
        newest_review=("review_date", "max"),
        avg_rating=("rating", "mean"),
        reviews_with_text=("review_text", lambda x: x.notna().sum()),
        reviews_with_app_version=("app_version", lambda x: x.notna().sum()),
        reviews_with_review_created_version=("review_created_version", lambda x: x.notna().sum()),
        reviews_with_developer_reply=("developer_reply", lambda x: x.notna().sum())
    )
    .reset_index()
)

google_validation_summary["duplicate_count"] = (
    google_validation_summary["review_count"] - google_validation_summary["unique_review_ids"]
)

google_validation_path = os.path.join(
    PROJECT_DIR,
    f"outputs/summaries/google_play_validation_summary_{RUN_LABEL}_{RUN_DATE}.csv"
)

google_validation_summary.to_csv(google_validation_path, index=False)

display(google_validation_summary)

print("Saved validation summary to:", google_validation_path)

,app_name,review_count,unique_review_ids,oldest_review,newest_review,avg_rating,reviews_with_text,reviews_with_app_version,reviews_with_review_created_version,reviews_with_developer_reply,duplicate_count
0,Spotify,2000,2000,2026-06-18 13:06:48,2026-06-21 02:49:56,3.5610,2000,1690,1690,298,0
1,TikTok,2000,2000,2026-06-18 08:14:37,2026-06-21 02:49:34,3.9700,2000,1290,1290,1652,0
2,YouTube,2000,2000,2026-06-19 14:22:31,2026-06-21 02:48:36,3.6435,2000,1959,1959,0,0


Saved validation summary to: /content/drive/MyDrive/app_review_source_validation/outputs/summaries/google_play_validation_summary_run1_2026_06_22.csv


## 3. Duplicate Handling

For repeated ingestion, each review needs a stable review key.

For Google Play, I use source, app name, country, language, and review ID.

For iOS, I use source, app name, country, and review ID.

This allows future runs to identify whether a review is new or already collected.

In [ ]:
def add_review_key(df):
    df = df.copy()

    for col in ["source", "app_name", "country", "review_id"]:
        if col not in df.columns:
            df[col] = ""

    if "language" not in df.columns:
        df["language"] = ""

    df["review_key"] = (
        df["source"].astype(str) + "_" +
        df["app_name"].astype(str) + "_" +
        df["country"].astype(str) + "_" +
        df["language"].astype(str) + "_" +
        df["review_id"].astype(str)
    )

    return df


google_reviews_checked = add_review_key(google_reviews)

google_duplicate_summary = pd.DataFrame([
    {
        "source": "google_play",
        "total_reviews": len(google_reviews_checked),
        "unique_review_keys": google_reviews_checked["review_key"].nunique(),
        "duplicate_rows": google_reviews_checked.duplicated("review_key").sum()
    }
])

google_duplicate_path = os.path.join(
    PROJECT_DIR,
    f"outputs/summaries/google_play_duplicate_summary_{RUN_LABEL}_{RUN_DATE}.csv"
)

google_duplicate_summary.to_csv(google_duplicate_path, index=False)

display(google_duplicate_summary)

print("Saved duplicate summary to:", google_duplicate_path)

,source,total_reviews,unique_review_keys,duplicate_rows
0,google_play,6000,6000,0


Saved duplicate summary to: /content/drive/MyDrive/app_review_source_validation/outputs/summaries/google_play_duplicate_summary_run1_2026_06_22.csv


## 4. iOS App Store Public Review Feed

iOS App Store reviews are tested using the public RSS review feed.

This path does not require App Store Connect owner access. However, the public feed has stronger limitations than Google Play, so this source is tested as a secondary path.

For Run 1, I tested multiple countries and pages to check whether country rotation and pagination can improve review coverage.

In [ ]:
def read_nested_value(obj, keys, default=None):
    value = obj

    for key in keys:
        if isinstance(value, dict):
            value = value.get(key)
        else:
            return default

    if value is None:
        return default

    return value


def collect_ios_reviews(app_list, country_list, max_pages=10):
    rows = []
    summary = []

    for app in app_list:
        app_name = app["name"]
        ios_id = app["ios_id"]

        for country in country_list:
            total = 0
            empty_pages = 0
            failed_pages = 0
            start = time.time()

            for page in range(1, max_pages + 1):
                url = (
                    f"https://itunes.apple.com/{country}/rss/customerreviews/"
                    f"page={page}/id={ios_id}/sortby=mostrecent/json"
                )

                try:
                    response = requests.get(url, timeout=20)

                    if response.status_code != 200:
                        failed_pages += 1
                        continue

                    data = response.json()
                    entries = data.get("feed", {}).get("entry", [])

                    if isinstance(entries, dict):
                        entries = [entries]

                    review_entries = [
                        e for e in entries
                        if "im:rating" in e and "content" in e
                    ]

                    if len(review_entries) == 0:
                        empty_pages += 1
                        continue

                    for r in review_entries:
                        rows.append({
                            "source": "ios_app_store",
                            "app_name": app_name,
                            "ios_id": ios_id,
                            "country": country,
                            "page": page,
                            "review_id": read_nested_value(r, ["id", "label"]),
                            "user_name": read_nested_value(r, ["author", "name", "label"]),
                            "rating": read_nested_value(r, ["im:rating", "label"]),
                            "title": read_nested_value(r, ["title", "label"]),
                            "review_text": read_nested_value(r, ["content", "label"]),
                            "app_version": read_nested_value(r, ["im:version", "label"]),
                            "review_date": read_nested_value(r, ["updated", "label"]),
                            "review_url": read_nested_value(r, ["link", "attributes", "href"])
                        })

                    total += len(review_entries)
                    time.sleep(0.3)

                except Exception:
                    failed_pages += 1
                    continue

            runtime_seconds = round(time.time() - start, 2)

            summary.append({
                "source": "ios_app_store",
                "app_name": app_name,
                "ios_id": ios_id,
                "country": country,
                "pages_requested": max_pages,
                "collected_count": total,
                "empty_pages": empty_pages,
                "failed_pages": failed_pages,
                "runtime_seconds": runtime_seconds
            })

    return pd.DataFrame(rows), pd.DataFrame(summary)

In [ ]:
ios_reviews, ios_collection_summary = collect_ios_reviews(apps, countries, max_pages=10)

ios_raw_path = os.path.join(
    PROJECT_DIR,
    f"data/raw/ios_reviews_{RUN_LABEL}_{RUN_DATE}.csv"
)

ios_summary_path = os.path.join(
    PROJECT_DIR,
    f"outputs/summaries/ios_collection_summary_{RUN_LABEL}_{RUN_DATE}.csv"
)

ios_reviews.to_csv(ios_raw_path, index=False)
ios_collection_summary.to_csv(ios_summary_path, index=False)

display(ios_collection_summary)

print("Saved iOS raw data to:", ios_raw_path)
print("Saved iOS summary to:", ios_summary_path)
print("Total iOS reviews collected:", len(ios_reviews))

,source,app_name,ios_id,country,pages_requested,collected_count,empty_pages,failed_pages,runtime_seconds
0,ios_app_store,YouTube,544007664,us,10,500,0,0,6.93
1,ios_app_store,YouTube,544007664,gb,10,500,0,0,5.57
2,ios_app_store,YouTube,544007664,ca,10,500,0,0,5.52
3,ios_app_store,YouTube,544007664,au,10,500,0,0,5.26
4,ios_app_store,YouTube,544007664,in,10,500,0,0,4.84
5,ios_app_store,TikTok,835599320,us,10,500,0,0,7.70
6,ios_app_store,TikTok,835599320,gb,10,500,0,0,5.76
7,ios_app_store,TikTok,835599320,ca,10,500,0,0,5.43
8,ios_app_store,TikTok,835599320,au,10,500,0,0,5.29
9,ios_app_store,TikTok,835599320,in,10,0,10,0,1.44


Saved iOS raw data to: /content/drive/MyDrive/app_review_source_validation/data/raw/ios_reviews_run1_2026_06_22.csv
Saved iOS summary to: /content/drive/MyDrive/app_review_source_validation/outputs/summaries/ios_collection_summary_run1_2026_06_22.csv
Total iOS reviews collected: 7000


## 5. iOS Validation Summary

This section summarizes the iOS App Store public RSS Run 1 result.

The goal is to check country/page coverage, review volume, empty pages, failed pages, and basic metadata availability.

Compared with Google Play, iOS requires country and page rotation to increase review coverage.

In [ ]:
ios_validation_summary = (
    ios_reviews
    .groupby(["app_name", "country"])
    .agg(
        review_count=("review_id", "count"),
        unique_review_ids=("review_id", "nunique"),
        oldest_review=("review_date", "min"),
        newest_review=("review_date", "max"),
        reviews_with_text=("review_text", lambda x: x.notna().sum()),
        reviews_with_app_version=("app_version", lambda x: x.notna().sum())
    )
    .reset_index()
)

ios_validation_summary["duplicate_count"] = (
    ios_validation_summary["review_count"] - ios_validation_summary["unique_review_ids"]
)

ios_validation_path = os.path.join(
    PROJECT_DIR,
    f"outputs/summaries/ios_validation_summary_{RUN_LABEL}_{RUN_DATE}.csv"
)

ios_validation_summary.to_csv(ios_validation_path, index=False)

display(ios_validation_summary)

print("Saved iOS validation summary to:", ios_validation_path)

,app_name,country,review_count,unique_review_ids,oldest_review,newest_review,reviews_with_text,reviews_with_app_version,duplicate_count
0,Spotify,au,500,500,2026-06-04T21:32:57-07:00,2026-06-20T15:41:55-07:00,500,500,0
1,Spotify,ca,500,500,2026-06-09T10:31:20-07:00,2026-06-20T15:09:11-07:00,500,500,0
2,Spotify,gb,500,500,2026-06-10T10:34:04-07:00,2026-06-20T15:54:20-07:00,500,500,0
3,Spotify,in,500,500,2026-06-05T07:28:30-07:00,2026-06-20T15:48:11-07:00,500,500,0
4,Spotify,us,500,500,2026-06-19T11:29:28-07:00,2026-06-20T15:55:38-07:00,500,500,0
5,TikTok,au,500,452,2026-02-22T07:53:39-07:00,2026-06-12T03:46:26-07:00,500,500,48
6,TikTok,ca,500,451,2026-04-18T08:02:17-07:00,2026-06-20T12:22:31-07:00,500,500,49
7,TikTok,gb,500,496,2026-05-28T17:15:42-07:00,2026-06-20T14:41:14-07:00,500,500,4
8,TikTok,us,500,500,2026-06-04T17:51:10-07:00,2026-06-20T15:57:12-07:00,500,500,0
9,YouTube,au,500,500,2026-04-30T17:53:58-07:00,2026-06-20T14:24:32-07:00,500,500,0


Saved iOS validation summary to: /content/drive/MyDrive/app_review_source_validation/outputs/summaries/ios_validation_summary_run1_2026_06_22.csv


In [ ]:
ios_reviews_checked = add_review_key(ios_reviews)

ios_duplicate_summary = pd.DataFrame([
    {
        "source": "ios_app_store",
        "total_reviews": len(ios_reviews_checked),
        "unique_review_keys": ios_reviews_checked["review_key"].nunique(),
        "duplicate_rows": ios_reviews_checked.duplicated("review_key").sum()
    }
])

ios_duplicate_path = os.path.join(
    PROJECT_DIR,
    f"outputs/summaries/ios_duplicate_summary_{RUN_LABEL}_{RUN_DATE}.csv"
)

ios_duplicate_summary.to_csv(ios_duplicate_path, index=False)

display(ios_duplicate_summary)

print("Saved iOS duplicate summary to:", ios_duplicate_path)

,source,total_reviews,unique_review_keys,duplicate_rows
0,ios_app_store,7000,6899,101


Saved iOS duplicate summary to: /content/drive/MyDrive/app_review_source_validation/outputs/summaries/ios_duplicate_summary_run1_2026_06_22.csv


## 6. Source-Level Comparison

This section compares Google Play and iOS App Store public review sources at a high level.

The comparison focuses on collected volume, duplicate rate, failed requests, and coverage limitations.

In [ ]:
source_comparison = pd.DataFrame([
    {
        "source": "google_play",
        "total_reviews": len(google_reviews_checked),
        "unique_review_keys": google_reviews_checked["review_key"].nunique(),
        "duplicate_rows": google_reviews_checked.duplicated("review_key").sum(),
        "failed_requests_or_apps": google_collection_summary["failed"].sum(),
        "main_limitation": "Needs repeated runs to confirm freshness and new review capture"
    },
    {
        "source": "ios_app_store",
        "total_reviews": len(ios_reviews_checked),
        "unique_review_keys": ios_reviews_checked["review_key"].nunique(),
        "duplicate_rows": ios_reviews_checked.duplicated("review_key").sum(),
        "failed_requests_or_apps": ios_collection_summary["failed_pages"].sum(),
        "main_limitation": "Country/page coverage can be inconsistent; some app-country pairs may return empty pages"
    }
])

source_comparison["duplicate_rate"] = (
    source_comparison["duplicate_rows"] / source_comparison["total_reviews"]
)

source_comparison_path = os.path.join(
    PROJECT_DIR,
    f"outputs/summaries/source_comparison_{RUN_LABEL}_{RUN_DATE}.csv"
)

source_comparison.to_csv(source_comparison_path, index=False)

display(source_comparison)

print("Saved source comparison to:", source_comparison_path)

,source,total_reviews,unique_review_keys,duplicate_rows,failed_requests_or_apps,main_limitation,duplicate_rate
0,google_play,6000,6000,0,0,Needs repeated runs to confirm freshness and n...,0.000000
1,ios_app_store,7000,6899,101,0,Country/page coverage can be inconsistent; som...,0.014429


Saved source comparison to: /content/drive/MyDrive/app_review_source_validation/outputs/summaries/source_comparison_run1_2026_06_22.csv


## 7. Exploratory and Data Quality Analysis

This section looks at the collected review data itself, not only whether the collection script works.

The checks include:

- rating distribution
- review length
- missing fields
- app version coverage
- detected language
- potential data quality issues

Google Play is still the main focus, but I also include iOS where the field structure is comparable.

In [ ]:
google_rating_distribution = (
    google_reviews_checked
    .groupby(["source", "app_name", "rating"])
    .size()
    .reset_index(name="review_count")
)

ios_reviews_checked["rating"] = pd.to_numeric(ios_reviews_checked["rating"], errors="coerce")

ios_rating_distribution = (
    ios_reviews_checked
    .groupby(["source", "app_name", "rating"])
    .size()
    .reset_index(name="review_count")
)

rating_distribution = pd.concat(
    [google_rating_distribution, ios_rating_distribution],
    ignore_index=True
)

rating_distribution_path = os.path.join(
    PROJECT_DIR,
    f"outputs/summaries/rating_distribution_{RUN_LABEL}_{RUN_DATE}.csv"
)

rating_distribution.to_csv(rating_distribution_path, index=False)

display(rating_distribution)

print("Saved rating distribution to:", rating_distribution_path)

,source,app_name,rating,review_count
0,google_play,Spotify,1,501
1,google_play,Spotify,2,148
2,google_play,Spotify,3,127
3,google_play,Spotify,4,176
4,google_play,Spotify,5,1048
5,google_play,TikTok,1,352
6,google_play,TikTok,2,97
7,google_play,TikTok,3,109
8,google_play,TikTok,4,143
9,google_play,TikTok,5,1299


Saved rating distribution to: /content/drive/MyDrive/app_review_source_validation/outputs/summaries/rating_distribution_run1_2026_06_22.csv


In [ ]:
def add_review_length(df):
    df = df.copy()
    df["review_text"] = df["review_text"].fillna("")
    df["review_char_length"] = df["review_text"].str.len()
    df["review_word_length"] = df["review_text"].str.split().str.len()
    return df


google_reviews_checked = add_review_length(google_reviews_checked)
ios_reviews_checked = add_review_length(ios_reviews_checked)

google_review_length_summary = (
    google_reviews_checked
    .groupby(["source", "app_name"])
    .agg(
        avg_char_length=("review_char_length", "mean"),
        median_char_length=("review_char_length", "median"),
        avg_word_length=("review_word_length", "mean"),
        median_word_length=("review_word_length", "median")
    )
    .reset_index()
)

ios_review_length_summary = (
    ios_reviews_checked
    .groupby(["source", "app_name"])
    .agg(
        avg_char_length=("review_char_length", "mean"),
        median_char_length=("review_char_length", "median"),
        avg_word_length=("review_word_length", "mean"),
        median_word_length=("review_word_length", "median")
    )
    .reset_index()
)

review_length_summary = pd.concat(
    [google_review_length_summary, ios_review_length_summary],
    ignore_index=True
)

review_length_path = os.path.join(
    PROJECT_DIR,
    f"outputs/summaries/review_length_summary_{RUN_LABEL}_{RUN_DATE}.csv"
)

review_length_summary.to_csv(review_length_path, index=False)

display(review_length_summary)

print("Saved review length summary to:", review_length_path)

,source,app_name,avg_char_length,median_char_length,avg_word_length,median_word_length
0,google_play,Spotify,86.9080,39.0,16.5885,8.0
1,google_play,TikTok,53.5170,20.0,10.4105,4.0
2,google_play,YouTube,60.3410,19.0,11.3370,4.0
3,ios_app_store,Spotify,96.1556,52.0,18.4048,10.0
4,ios_app_store,TikTok,176.7205,125.0,33.7430,24.0
5,ios_app_store,YouTube,139.3120,72.0,25.5508,14.0


Saved review length summary to: /content/drive/MyDrive/app_review_source_validation/outputs/summaries/review_length_summary_run1_2026_06_22.csv


In [ ]:
google_fields_to_check = [
    "review_id",
    "user_name",
    "review_text",
    "rating",
    "thumbs_up",
    "review_created_version",
    "review_date",
    "developer_reply",
    "reply_date",
    "app_version"
]

google_missing_summary = (
    google_reviews_checked[google_fields_to_check]
    .isna()
    .mean()
    .reset_index()
)

google_missing_summary.columns = ["field", "missing_rate"]
google_missing_summary["source"] = "google_play"


ios_fields_to_check = [
    "review_id",
    "user_name",
    "review_text",
    "rating",
    "title",
    "app_version",
    "review_date",
    "review_url"
]

ios_missing_summary = (
    ios_reviews_checked[ios_fields_to_check]
    .isna()
    .mean()
    .reset_index()
)

ios_missing_summary.columns = ["field", "missing_rate"]
ios_missing_summary["source"] = "ios_app_store"


missing_summary = pd.concat(
    [google_missing_summary, ios_missing_summary],
    ignore_index=True
)

missing_summary = missing_summary[["source", "field", "missing_rate"]]

missing_summary_path = os.path.join(
    PROJECT_DIR,
    f"outputs/summaries/missing_summary_{RUN_LABEL}_{RUN_DATE}.csv"
)

missing_summary.to_csv(missing_summary_path, index=False)

display(missing_summary)

print("Saved missing summary to:", missing_summary_path)

,source,field,missing_rate
0,google_play,review_id,0.000000
1,google_play,user_name,0.000000
2,google_play,review_text,0.000000
3,google_play,rating,0.000000
4,google_play,thumbs_up,0.000000
5,google_play,review_created_version,0.176833
6,google_play,review_date,0.000000
7,google_play,developer_reply,0.675000
8,google_play,reply_date,0.675000
9,google_play,app_version,0.176833


Saved missing summary to: /content/drive/MyDrive/app_review_source_validation/outputs/summaries/missing_summary_run1_2026_06_22.csv


In [ ]:
google_app_version_coverage = (
    google_reviews_checked
    .groupby(["source", "app_name"])
    .agg(
        total_reviews=("review_id", "count"),
        reviews_with_app_version=("app_version", lambda x: x.notna().sum())
    )
    .reset_index()
)

ios_app_version_coverage = (
    ios_reviews_checked
    .groupby(["source", "app_name"])
    .agg(
        total_reviews=("review_id", "count"),
        reviews_with_app_version=("app_version", lambda x: x.notna().sum())
    )
    .reset_index()
)

app_version_coverage = pd.concat(
    [google_app_version_coverage, ios_app_version_coverage],
    ignore_index=True
)

app_version_coverage["app_version_coverage_rate"] = (
    app_version_coverage["reviews_with_app_version"] / app_version_coverage["total_reviews"]
)

app_version_path = os.path.join(
    PROJECT_DIR,
    f"outputs/summaries/app_version_coverage_{RUN_LABEL}_{RUN_DATE}.csv"
)

app_version_coverage.to_csv(app_version_path, index=False)

display(app_version_coverage)

print("Saved app version coverage to:", app_version_path)

,source,app_name,total_reviews,reviews_with_app_version,app_version_coverage_rate
0,google_play,Spotify,2000,1690,0.8450
1,google_play,TikTok,2000,1290,0.6450
2,google_play,YouTube,2000,1959,0.9795
3,ios_app_store,Spotify,2500,2500,1.0000
4,ios_app_store,TikTok,2000,2000,1.0000
5,ios_app_store,YouTube,2500,2500,1.0000


Saved app version coverage to: /content/drive/MyDrive/app_review_source_validation/outputs/summaries/app_version_coverage_run1_2026_06_22.csv


In [ ]:
def safe_detect_language(text):
    try:
        if pd.isna(text) or str(text).strip() == "":
            return "unknown"
        return detect(str(text))
    except:
        return "unknown"


google_reviews_checked["detected_language"] = google_reviews_checked["review_text"].apply(safe_detect_language)
ios_reviews_checked["detected_language"] = ios_reviews_checked["review_text"].apply(safe_detect_language)

google_language_summary = (
    google_reviews_checked
    .groupby(["source", "app_name", "detected_language"])
    .size()
    .reset_index(name="review_count")
)

ios_language_summary = (
    ios_reviews_checked
    .groupby(["source", "app_name", "detected_language"])
    .size()
    .reset_index(name="review_count")
)

language_summary = pd.concat(
    [google_language_summary, ios_language_summary],
    ignore_index=True
)

language_summary = language_summary.sort_values(
    ["source", "app_name", "review_count"],
    ascending=[True, True, False]
)

language_path = os.path.join(
    PROJECT_DIR,
    f"outputs/summaries/language_summary_{RUN_LABEL}_{RUN_DATE}.csv"
)

language_summary.to_csv(language_path, index=False)

display(language_summary)

print("Saved language summary to:", language_path)

,source,app_name,detected_language,review_count
8,google_play,Spotify,en,1425
30,google_play,Spotify,so,93
0,google_play,Spotify,af,58
26,google_play,Spotify,ro,39
29,google_play,Spotify,sl,36
...,...,...,...,...
200,ios_app_store,YouTube,hi,1
206,ios_app_store,YouTube,mk,1
207,ios_app_store,YouTube,ml,1
209,ios_app_store,YouTube,ne,1


Saved language summary to: /content/drive/MyDrive/app_review_source_validation/outputs/summaries/language_summary_run1_2026_06_22.csv


In [ ]:
google_checked_path = os.path.join(
    PROJECT_DIR,
    f"data/processed/google_play_reviews_checked_{RUN_LABEL}_{RUN_DATE}.csv"
)

ios_checked_path = os.path.join(
    PROJECT_DIR,
    f"data/processed/ios_reviews_checked_{RUN_LABEL}_{RUN_DATE}.csv"
)

google_reviews_checked.to_csv(google_checked_path, index=False)
ios_reviews_checked.to_csv(ios_checked_path, index=False)

print("Saved Google Play checked data to:", google_checked_path)
print("Saved iOS checked data to:", ios_checked_path)

Saved Google Play checked data to: /content/drive/MyDrive/app_review_source_validation/data/processed/google_play_reviews_checked_run1_2026_06_22.csv
Saved iOS checked data to: /content/drive/MyDrive/app_review_source_validation/data/processed/ios_reviews_checked_run1_2026_06_22.csv


## 8. Run 1 Findings

This clean Run 1 validation tested Google Play and iOS App Store public review sources for recurring review ingestion.

### Google Play Result

Google Play performed strongly in Run 1.

The collection returned 6,000 total reviews across three apps: YouTube, TikTok, and Spotify. Each app returned 2,000 newest US / English reviews. All three app requests completed successfully, with no failed app-level requests.

Google Play also provided stable review IDs. The duplicate check showed 6,000 total reviews, 6,000 unique review keys, and 0 duplicate rows. This is important because stable review IDs make future repeated collection and duplicate handling much easier.

The metadata coverage was also usable. Core fields such as review ID, user name, review text, rating, thumbs-up count, and review date were complete. App version fields were partially missing, and developer reply fields were missing for many reviews, but this is expected because not every review has an app version or developer response.

Based on Run 1, Google Play is the stronger primary source for repeated collection testing.

### iOS App Store Result

The iOS App Store public RSS feed was also technically accessible.

The Run 1 iOS test requested reviews for three apps across five countries and ten pages per app-country pair. The test collected 7,000 total reviews. Most app-country pairs returned usable results, but TikTok India returned 0 reviews with 10 empty pages.

This shows that iOS country and page rotation can increase review coverage, but the coverage is still less consistent than Google Play. Some app-country pairs may return empty results even when the request itself does not fail.

The iOS duplicate check showed 7,000 total reviews, 6,899 unique review keys, and 101 duplicate rows. The duplicate rate was about 1.44%. This is manageable, but it confirms that duplicate handling is necessary for iOS.

### Source Comparison

Google Play and iOS both passed the basic technical access test, but they have different strengths.

Google Play is cleaner for recurring ingestion because it returned stable review IDs, no duplicate rows in Run 1, strong volume, and consistent metadata.

iOS returned even more total reviews in this run, but it required country/page rotation and still had empty coverage for one app-country pair. iOS also had some duplicate rows, so it is better treated as a secondary source for now.

### Exploratory and Data Quality Findings

I also started exploratory and data quality analysis on the collected data.

The rating distribution shows that both sources contain a mix of positive and negative reviews, with many 1-star and 5-star reviews. This is useful for downstream sentiment or quality analysis.

The review length summary shows that iOS reviews are generally longer than Google Play reviews in this run. For example, TikTok iOS reviews had a higher average and median review length than TikTok Google Play reviews.

The missing field summary shows that Google Play core fields were complete, but app version and developer reply fields had missing values. iOS fields checked in this run were complete for the collected reviews.

The app version coverage check shows that iOS had full app version coverage in this run, while Google Play app version coverage varied by app.

The language detection result suggests that most reviews are English, but some reviews were detected as other languages. Since language detection can be noisy for short reviews, this should be treated as a rough data quality flag rather than a perfect language label.

### Current Conclusion

Run 1 confirms that Google Play is feasible as the primary source for a recurring app review ingestion pilot.

iOS App Store public RSS feed is also feasible as a secondary source, but it has more coverage and duplicate-handling issues.

The next step is to run the same notebook again as Run 2 and compare Run 2 against Run 1. The Run 2 comparison should focus on new review counts, overlap with Run 1, duplicate rates across runs, field consistency, request failures, and freshness patterns.

In [ ]:
run1_report = f"""
# Run 1 Findings

## Purpose

This clean Run 1 validation tested Google Play and iOS App Store public review sources for recurring review ingestion.

The validation focused on:

- access method
- review volume
- available metadata fields
- pagination and batching
- duplicate handling
- freshness
- field consistency
- request failures
- exploratory analysis
- data quality issues

## Google Play Result

Google Play performed strongly in Run 1.

The collection returned 6,000 total reviews across three apps: YouTube, TikTok, and Spotify. Each app returned 2,000 newest US / English reviews. All three app requests completed successfully, with no failed app-level requests.

Google Play also provided stable review IDs. The duplicate check showed 6,000 total reviews, 6,000 unique review keys, and 0 duplicate rows. This is important because stable review IDs make future repeated collection and duplicate handling much easier.

The metadata coverage was also usable. Core fields such as review ID, user name, review text, rating, thumbs-up count, and review date were complete. App version fields were partially missing, and developer reply fields were missing for many reviews, but this is expected because not every review has an app version or developer response.

Based on Run 1, Google Play is the stronger primary source for repeated collection testing.

## iOS App Store Result

The iOS App Store public RSS feed was also technically accessible.

The Run 1 iOS test requested reviews for three apps across five countries and ten pages per app-country pair. The test collected 7,000 total reviews. Most app-country pairs returned usable results, but TikTok India returned 0 reviews with 10 empty pages.

This shows that iOS country and page rotation can increase review coverage, but the coverage is still less consistent than Google Play. Some app-country pairs may return empty results even when the request itself does not fail.

The iOS duplicate check showed 7,000 total reviews, 6,899 unique review keys, and 101 duplicate rows. The duplicate rate was about 1.44%. This is manageable, but it confirms that duplicate handling is necessary for iOS.

## Source Comparison

Google Play and iOS both passed the basic technical access test, but they have different strengths.

Google Play is cleaner for recurring ingestion because it returned stable review IDs, no duplicate rows in Run 1, strong volume, and consistent metadata.

iOS returned even more total reviews in this run, but it required country/page rotation and still had empty coverage for one app-country pair. iOS also had some duplicate rows, so it is better treated as a secondary source for now.

## Exploratory and Data Quality Findings

I also started exploratory and data quality analysis on the collected data.

The rating distribution shows that both sources contain a mix of positive and negative reviews, with many 1-star and 5-star reviews. This is useful for downstream sentiment or quality analysis.

The review length summary shows that iOS reviews are generally longer than Google Play reviews in this run.

The missing field summary shows that Google Play core fields were complete, but app version and developer reply fields had missing values. iOS fields checked in this run were complete for the collected reviews.

The app version coverage check shows that iOS had full app version coverage in this run, while Google Play app version coverage varied by app.

The language detection result suggests that most reviews are English, but some reviews were detected as other languages. Since language detection can be noisy for short reviews, this should be treated as a rough data quality flag rather than a perfect language label.

## Current Conclusion

Run 1 confirms that Google Play is feasible as the primary source for a recurring app review ingestion pilot.

iOS App Store public RSS feed is also feasible as a secondary source, but it has more coverage and duplicate-handling issues.

The next step is to run the same notebook again as Run 2 and compare Run 2 against Run 1. The Run 2 comparison should focus on new review counts, overlap with Run 1, duplicate rates across runs, field consistency, request failures, and freshness patterns.
"""

run1_report_path = os.path.join(
    PROJECT_DIR,
    f"reports/run1_findings_{RUN_DATE}.md"
)

with open(run1_report_path, "w") as f:
    f.write(run1_report)

print("Saved Run 1 report to:", run1_report_path)

Saved Run 1 report to: /content/drive/MyDrive/app_review_source_validation/reports/run1_findings_2026_06_22.md
